# Local Model Smoke Test

This notebook tests the llama.cpp server integration and verifies that:
1. The model server is reachable
2. Text generation works
3. Tracing is functional
4. Token counting works

## Prerequisites

Before running this notebook:
1. Install dependencies: `pip install -r requirements.txt`
2. Install model server: See `install_day3.sh`
3. Start the model server: `bash scripts/start_model_server.sh`
4. (Optional) Start Phoenix: `phoenix serve`

In [1]:
# Setup
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: /Users/alistair/Documents/101_Coding/Projects/AgenticAI/agentic_workbench


In [2]:
# Imports
from workbench.core.config import load_config
from workbench.models import LlamaCppServerModel
from workbench.observability import setup_tracing, start_span

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Setup observability
setup_tracing(service_name="notebook-test")

# For notebook usage, we'll create a simple run context
# This is optional - the model works fine without it
from workbench.core.run_context import run_context
from workbench.core.config import create_config_snapshot

print("✓ Observability setup complete")
print("  If Phoenix is running, traces will appear at: http://localhost:6006")
print("\nNote: Model will work with or without full run context")

✓ Observability setup complete
  If Phoenix is running, traces will appear at: http://localhost:6006

Note: Model will work with or without full run context


In [4]:
# Load configuration
config = load_config()

print("Configuration:")
print(f"  Model server: {config.model['base_url']}")
print(f"  Model name: {config.model['model_name']}")
print(f"  Temperature: {config.model['temperature']}")
print(f"  Max tokens: {config.model['max_tokens']}")

Configuration:
  Model server: http://localhost:8080
  Model name: ministral-8b
  Temperature: 0.7
  Max tokens: 2048


In [5]:
# Create model instance
model = LlamaCppServerModel(
    base_url=config.model['base_url'],
    model_name=config.model['model_name'],
    timeout_seconds=config.model['timeout_seconds'],
    default_temperature=config.model['temperature'],
    default_max_tokens=config.model['max_tokens'],
)

print("✓ Model instance created")

✓ Model instance created


## Test 1: Simple Generation

Test basic text generation with a simple prompt.

In [6]:
# Test simple generation
messages = [
    {"role": "user", "content": "Say hello in one sentence."}
]

with start_span("simple_generation_test"):
    response = model.generate(messages, max_tokens=50, temperature=0.7)

print("Response:")
print(f"  Text: {response.text}")
print(f"  Tokens in: {response.tokens_in}")
print(f"  Tokens out: {response.tokens_out}")
print(f"  Latency: {response.latency_ms:.2f}ms")
print(f"  Finish reason: {response.finish_reason}")

Response:
  Text: Hello! How can I assist you today?
  Tokens in: 9
  Tokens out: 9
  Latency: 452.91ms
  Finish reason: stop


## Test 2: System Message

Test generation with a system message to guide behavior.

In [7]:
# Test with system message
messages = [
    {"role": "system", "content": "You are a helpful assistant that speaks in a friendly, casual tone."},
    {"role": "user", "content": "Explain what RAG is in one sentence."}
]

with start_span("system_message_test"):
    response = model.generate(messages, max_tokens=100, temperature=0.7)

print("Response:")
print(f"  {response.text}")
print(f"\n  Tokens: {response.tokens_out}")
print(f"  Latency: {response.latency_ms:.2f}ms")

Response:
  Sure! RAG stands for Retrieval-Augmented Generation. It's a technique that combines information retrieval with text generation to produce more accurate and relevant responses.

  Tokens: 32
  Latency: 983.76ms


## Test 3: Multi-turn Conversation

Test a multi-turn conversation to verify context handling.

In [8]:
# Test multi-turn conversation
messages = [
    {"role": "user", "content": "What is 2+2?"},
    {"role": "assistant", "content": "4"},
    {"role": "user", "content": "And if I add 3 to that?"},
]

with start_span("multiturn_test"):
    response = model.generate(messages, max_tokens=50, temperature=0.0)

print("Response:")
print(f"  {response.text}")
print(f"\n  Tokens: {response.tokens_out}")
print(f"  Latency: {response.latency_ms:.2f}ms")

Response:
  7

  Tokens: 1
  Latency: 224.35ms


## Test 4: Temperature Comparison

Compare outputs at different temperatures.

In [9]:
# Test temperature variation
prompt = "Tell me an interesting fact about artificial intelligence."
messages = [{"role": "user", "content": prompt}]

print("Temperature comparison:\n")

for temp in [0.0, 0.5, 1.0, 1.5]:
    response = model.generate(messages, max_tokens=100, temperature=temp)
    print(f"Temperature {temp}:")
    print(f"  {response.text[:150]}...")
    print()

Temperature comparison:

Temperature 0.0:
  Sure! One interesting fact about artificial intelligence is that it can be used to create art. AI has been trained on vast amounts of data, including ...

Temperature 0.5:
  Sure! One interesting fact about artificial intelligence (AI) is that AI can be used to predict earthquakes. Researchers have developed AI algorithms ...

Temperature 1.0:
  Absolutely! One interesting fact about artificial intelligence is that AI has been used to compose music. For instance, the AIVA (Artificial Intellige...

Temperature 1.5:
  An interesting fact about artificial intelligence is that it can help identify cancer from medical images with nearly as high an accuracy rate as huma...



## Test 5: Token Counting

Test token counting functionality.

In [10]:
# Test token counting
test_texts = [
    "Hello",
    "This is a test.",
    "This is a longer sentence with more words to count tokens.",
    "The quick brown fox jumps over the lazy dog. This classic pangram contains every letter of the English alphabet.",
]

print("Token counting:\n")
for text in test_texts:
    count = model.count_tokens(text)
    print(f"Text: {text[:50]}{'...' if len(text) > 50 else ''}")
    print(f"  Estimated tokens: {count}")
    print(f"  Chars/token: {len(text)/count:.2f}")
    print()

Token counting:

Text: Hello
  Estimated tokens: 1
  Chars/token: 5.00

Text: This is a test.
  Estimated tokens: 3
  Chars/token: 5.00

Text: This is a longer sentence with more words to count...
  Estimated tokens: 14
  Chars/token: 4.14

Text: The quick brown fox jumps over the lazy dog. This ...
  Estimated tokens: 28
  Chars/token: 4.00



## Test 6: Interactive Chat

Have an interactive conversation with the model.

In [11]:
# Interactive chat
conversation = []

def chat(user_message: str, max_tokens: int = 200, temperature: float = 0.7):
    """Send a message and get a response."""
    conversation.append({"role": "user", "content": user_message})
    
    response = model.generate(conversation, max_tokens=max_tokens, temperature=temperature)
    
    conversation.append({"role": "assistant", "content": response.text})
    
    print(f"User: {user_message}")
    print(f"Assistant: {response.text}")
    print(f"  [{response.tokens_out} tokens, {response.latency_ms:.0f}ms]\n")
    
    return response

# Start conversation
print("Interactive Chat Session\n" + "=" * 50 + "\n")

Interactive Chat Session



In [12]:
# First message
chat("Hi! What's your name?")

User: Hi! What's your name?
Assistant: Hello! I don't have a name, but you can give me one if you'd like. How about you? What's your name?
  [29 tokens, 919ms]



ModelResponse(text="Hello! I don't have a name, but you can give me one if you'd like. How about you? What's your name?", tokens_in=10, tokens_out=29, latency_ms=918.9887046813965, finish_reason='stop', metadata={'model': 'ministral-8b', 'provider': 'llamacpp_server', 'raw_usage': {'prompt_tokens': 10, 'completion_tokens': 29, 'total_tokens': 39}})

In [13]:
# Continue conversation
chat("Can you help me understand how RAG systems work?")

User: Can you help me understand how RAG systems work?
Assistant: Absolutely, I'd be happy to help explain Retrieval-Augmented Generation (RAG) systems!

### Overview of RAG Systems

Retrieval-Augmented Generation is a technique that combines the strengths of two different approaches: information retrieval and text generation. Here's how it works:

1. **Information Retrieval**:
   - When a user asks a question or makes a request, a system first retrieves relevant documents or passages from a large corpus of data.
   - These documents are typically stored in a database or an index that is optimized for fast retrieval.

2. **Text Generation**:
   - Once the relevant documents are retrieved, the system uses a language model (like me!) to generate a response based on both the user's input and the content from the retrieved documents.
   - The language model is trained to understand context and generate coherent responses that address the original query.

### Key Components

1. **Retrieval 

ModelResponse(text="Absolutely, I'd be happy to help explain Retrieval-Augmented Generation (RAG) systems!\n\n### Overview of RAG Systems\n\nRetrieval-Augmented Generation is a technique that combines the strengths of two different approaches: information retrieval and text generation. Here's how it works:\n\n1. **Information Retrieval**:\n   - When a user asks a question or makes a request, a system first retrieves relevant documents or passages from a large corpus of data.\n   - These documents are typically stored in a database or an index that is optimized for fast retrieval.\n\n2. **Text Generation**:\n   - Once the relevant documents are retrieved, the system uses a language model (like me!) to generate a response based on both the user's input and the content from the retrieved documents.\n   - The language model is trained to understand context and generate coherent responses that address the original query.\n\n### Key Components\n\n1. **Retrieval Module**:\n   - This module is

In [14]:
# Add your own messages here
chat("What are the main components of a RAG system?")

User: What are the main components of a RAG system?
Assistant: The main components of a Retrieval-Augmented Generation (RAG) system can be broken down into several key parts. Here they are:

1. **Retrieval Module**:
   - This module is responsible for finding relevant documents or passages from a large corpus based on the user's query.
   - It uses techniques such as keyword matching, vector similarity search, and other retrieval strategies to fetch the most pertinent information.

2. **Document Indexing**:
   - Before retrieval can happen, documents must be indexed using a technique like TF-IDF (Term Frequency-Inverse Document Frequency) or more advanced methods like BERT embeddings.
   - The index is used by the retrieval module to quickly find relevant documents.

3. **Language Model**:
   - This component takes the user's query and the retrieved documents as inputs and generates a coherent response.
   - The language model might be based on transformer architectures (like BERT or T

ModelResponse(text="The main components of a Retrieval-Augmented Generation (RAG) system can be broken down into several key parts. Here they are:\n\n1. **Retrieval Module**:\n   - This module is responsible for finding relevant documents or passages from a large corpus based on the user's query.\n   - It uses techniques such as keyword matching, vector similarity search, and other retrieval strategies to fetch the most pertinent information.\n\n2. **Document Indexing**:\n   - Before retrieval can happen, documents must be indexed using a technique like TF-IDF (Term Frequency-Inverse Document Frequency) or more advanced methods like BERT embeddings.\n   - The index is used by the retrieval module to quickly find relevant documents.\n\n3. **Language Model**:\n   - This component takes the user's query and the retrieved documents as inputs and generates a coherent response.\n   - The language model might be based on transformer architectures (like BERT or T5) that are trained for natural

## Summary

If all cells above executed successfully, your setup is working! ✓

### Next Steps:

1. Check Phoenix UI (if running): http://localhost:6006
   - You should see traces for each generation
   - Spans should show timing and token counts

2. Run the unit tests:
   ```bash
   python -m pytest tests/test_language_model_unit.py -v
   ```

3. Run the integration tests:
   ```bash
   python -m pytest tests/test_language_model_integration.py -v -m integration
   ```

4. Experiment with different prompts and settings in this notebook!